# Temporal Feature Engineering

This notebook performs feature engineering for the temporal congestion model.

The lag features implemented in this stage were motivated by the temporal exploratory analysis conducted in the previous notebook. The analysis suggested that operational disruptions in port logistics systems may propagate across operational cycles.

To capture these delayed effects, lagged versions of key operational indicators are engineered.

The resulting dataset will be used in the subsequent feature selection and model training stages.

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../../data/JNPA_Cleaned_Data.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

df.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,Exp_Transit_CFS,Exp_Transit_ICD,Total_TEUs_Handled,Date
0,Jan 23,21.9,18.2,86.3,1.88,2.86,83.2,70.9,70.5,73.4,4.45,107.6,522592,2023-01-01
1,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,5.06,109.2,467614,2023-02-01
2,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,6.42,113.0,560076,2023-03-01
3,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,6.46,89.3,503668,2023-04-01
4,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,4.21,109.6,538247,2023-05-01


## Creating Operational Indicators

Some operational indicators used in the temporal model are derived variables rather than raw observations.

Rail Friction represents the imbalance between rail evacuation dwell time and truck evacuation dwell time.

Stress represents the interaction between container throughput and seasonal disruption periods such as monsoon months.

In [2]:
df["Rail_Friction"] = df["Imp_Dwell_Train"] / df["Imp_Dwell_Truck"]

df["Is_Monsoon"] = df["Date"].dt.month.isin([6,7,8,9]).astype(int)

df["Stress"] = df["Total_TEUs_Handled"] * df["Is_Monsoon"]

## Creating Congestion Risk Indicator

A congestion risk indicator is created based on the 75th percentile of import dwell time.

This threshold acts as an early warning signal for congestion conditions rather than waiting for the operational demurrage threshold of 72 hours.

Months with import dwell values above this percentile are flagged as congestion events.

In [3]:
threshold = df["Imp_Dwell_Overall"].quantile(0.75)

df["Risk"] = (df["Imp_Dwell_Overall"] > threshold).astype(int)

print("Risk threshold:", threshold)

df["Risk"].value_counts()

Risk threshold: 27.15


Risk
0    26
1     9
Name: count, dtype: int64

## Engineering Temporal Lag Features

Based on the temporal congestion hypothesis, lagged versions of key operational indicators are created.

These lag variables represent operational conditions in the previous month and allow the model to capture delayed congestion effects.

The following lag features are engineered:

Volume_Lag1 – previous-month container throughput

Parking_Dwell_Lag1 – previous-month gate congestion indicator

Rail_Friction_Lag1 – previous-month rail evacuation imbalance

Stress_Lag1 – previous-month seasonal operational pressure

In [4]:
df["Volume_Lag1"] = df["Total_TEUs_Handled"].shift(1)

df["Parking_Dwell_Lag1"] = df["Parking_Dwell"].shift(1)

df["Rail_Friction_Lag1"] = df["Rail_Friction"].shift(1)

df["Stress_Lag1"] = df["Stress"].shift(1)

## Handling Missing Values Created by Lag Operations

The shift operation introduces missing values for the first observation since no previous-month data exists.

These rows are removed to maintain dataset consistency for model training.

In [5]:
df = df.dropna().reset_index(drop=True)

df.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,...,Total_TEUs_Handled,Date,Rail_Friction,Is_Monsoon,Stress,Risk,Volume_Lag1,Parking_Dwell_Lag1,Rail_Friction_Lag1,Stress_Lag1
0,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,...,467614,2023-02-01,2.900000,0,0,1,522592.0,1.88,4.741758,0.0
1,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,...,560076,2023-03-01,2.856502,0,0,0,467614.0,2.55,2.900000,0.0
2,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,...,503668,2023-04-01,2.094488,0,0,1,560076.0,3.65,2.856502,0.0
3,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,...,538247,2023-05-01,2.965116,0,0,0,503668.0,4.00,2.094488,0.0
4,Jun 23,15.8,13.8,38.4,2.18,2.35,107.0,71.9,69.8,88.5,...,523948,2023-06-01,2.782609,1,523948,0,538247.0,2.33,2.965116,0.0


## Verifying Engineered Features

In [6]:
df[["Volume_Lag1","Parking_Dwell_Lag1","Rail_Friction_Lag1","Stress_Lag1"]].describe()

,Volume_Lag1,Parking_Dwell_Lag1,Rail_Friction_Lag1,Stress_Lag1
count,34.000000,34.000000,34.000000,34.000000
mean,587279.735294,1.751471,2.944374,213099.411765
std,58742.639188,0.799102,0.999253,295173.769516
min,467614.000000,0.570000,1.635854,0.000000
25%,543493.250000,1.100000,2.227899,0.000000
50%,580112.000000,1.775000,2.868042,0.000000
75%,634686.750000,2.160000,3.324109,548713.750000
max,716056.000000,4.000000,5.805031,716056.000000


## Saving Temporal Feature Dataset

The engineered dataset is saved for use in the feature selection and model training stages.

In [7]:
df.to_csv("../../data/JNPA_feature_engineered_temporal.csv", index=False)